# 04 - Data Cleaning and Validation

## Objective
Clean and validate the Texas CRIS crash data for Dallas and Tarrant counties from 2021 through 2025.

## Tasks
1. Inspect annual datasets and schema differences.
2. Validate crash identifiers, dates, and severity codes.
3. Investigate missing values and duplicate records.
4. Standardize data types and align schemas.
5. Create reproducible processed datasets and cleaning reports.

## Data integrity principles
- Preserve original raw datasets.
- Never invent severity-code meanings.
- Document every cleaning decision.
- Avoid target leakage in future predictive models.
- Do not remove records without a documented reason.

## Step 1 — Clone your GitHub repository

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_DIR = Path("/content/traffic-accident-risk-analytics-github")
RAW_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
REPORTS_DIR = PROJECT_DIR / "reports"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

years = [2021, 2022, 2023, 2024, 2025]

print("Project:", PROJECT_DIR)
print("Raw data:", RAW_DIR)
print("Processed data:", PROCESSED_DIR)
print("Reports:", REPORTS_DIR)

Project: /content/traffic-accident-risk-analytics-github
Raw data: /content/traffic-accident-risk-analytics-github/data/raw
Processed data: /content/traffic-accident-risk-analytics-github/data/processed
Reports: /content/traffic-accident-risk-analytics-github/reports


## Step 2 — Verify the raw data folders

In [ ]:
annual_data = {}
inventory = []

for year in years:
    file_path = RAW_DIR / str(year) / "crash.csv"

    if not file_path.exists():
        print(f"ERROR: Missing file for {year}")
        continue

    df = pd.read_csv(file_path, low_memory=False)
    annual_data[year] = df

    inventory.append({
        "year": year,
        "rows": len(df),
        "columns": len(df.columns),
        "duplicate_rows": int(df.duplicated().sum()),
        "file_size_MB": round(file_path.stat().st_size / (1024**2), 2)
    })

inventory_df = pd.DataFrame(inventory)
display(inventory_df)

print("Annual datasets loaded:", len(annual_data))

,year,rows,columns,duplicate_rows,file_size_MB
0,2021,97947,174,0,43.83
1,2022,94700,174,0,42.27
2,2023,92082,183,0,48.79
3,2024,83092,183,0,42.91
4,2025,82670,183,0,42.65


Annual datasets loaded: 5


## Step 3 — Compare annual schemas

In [ ]:
all_columns = set().union(
    *(set(df.columns) for df in annual_data.values())
)

shared_columns = set.intersection(
    *(set(df.columns) for df in annual_data.values())
)

print("Total distinct columns:", len(all_columns))
print("Columns shared across all years:", len(shared_columns))

schema_differences = []

for year, df in annual_data.items():
    missing_columns = sorted(all_columns - set(df.columns))
    extra_summary = {
        "year": year,
        "column_count": len(df.columns),
        "columns_missing_from_year": len(missing_columns),
        "missing_column_names": ", ".join(missing_columns)
    }
    schema_differences.append(extra_summary)

display(pd.DataFrame(schema_differences))

Total distinct columns: 183
Columns shared across all years: 174


,year,column_count,columns_missing_from_year,missing_column_names
0,2021,174,9,"Investigat_Arrv_Date, Investigat_Date_Rwy_Clrd..."
1,2022,174,9,"Investigat_Arrv_Date, Investigat_Date_Rwy_Clrd..."
2,2023,183,0,
3,2024,183,0,
4,2025,183,0,


## Step 4 — Identify the actual severity, date, and ID columns

In [ ]:
for year, df in annual_data.items():
    print(f"\n{'=' * 50}")
    print("YEAR:", year)

    print("Crash ID columns:")
    print([c for c in df.columns if "crash_id" in c.lower()])

    print("Severity-related columns:")
    print([
        c for c in df.columns
        if "severity" in c.lower() or "crash_sev" in c.lower()
    ])

    print("Date-related columns:")
    print([c for c in df.columns if "crash_date" in c.lower()])

    print("Time-related columns:")
    print([c for c in df.columns if "crash_time" in c.lower()])


YEAR: 2021
Crash ID columns:
['Crash_ID']
Severity-related columns:
['Crash_Sev_ID']
Date-related columns:
['Crash_Date']
Time-related columns:
['Crash_Time']

YEAR: 2022
Crash ID columns:
['Crash_ID']
Severity-related columns:
['Crash_Sev_ID']
Date-related columns:
['Crash_Date']
Time-related columns:
['Crash_Time']

YEAR: 2023
Crash ID columns:
['Crash_ID']
Severity-related columns:
['Crash_Sev_ID']
Date-related columns:
['Crash_Date']
Time-related columns:
['Crash_Time']

YEAR: 2024
Crash ID columns:
['Crash_ID']
Severity-related columns:
['Crash_Sev_ID']
Date-related columns:
['Crash_Date']
Time-related columns:
['Crash_Time']

YEAR: 2025
Crash ID columns:
['Crash_ID']
Severity-related columns:
['Crash_Sev_ID']
Date-related columns:
['Crash_Date']
Time-related columns:
['Crash_Time']


## Step 5 — Check severity values

In [ ]:
severity_results = []

for year, df in annual_data.items():
    severity_cols = [
        c for c in df.columns
        if "severity" in c.lower() or "crash_sev" in c.lower()
    ]

    for col in severity_cols:
        counts = df[col].value_counts(dropna=False)

        for value, count in counts.items():
            severity_results.append({
                "year": year,
                "column": col,
                "value": str(value),
                "count": int(count)
            })

severity_df = pd.DataFrame(severity_results)
display(severity_df)

,year,column,value,count
0,2021,Crash_Sev_ID,5,59573
1,2021,Crash_Sev_ID,3,16235
2,2021,Crash_Sev_ID,2,11396
3,2021,Crash_Sev_ID,0,7562
4,2021,Crash_Sev_ID,1,2631
5,2021,Crash_Sev_ID,4,550
6,2022,Crash_Sev_ID,5,57449
7,2022,Crash_Sev_ID,3,14780
8,2022,Crash_Sev_ID,2,12255
9,2022,Crash_Sev_ID,0,7163


## Step 6 — Save the initial validation report

In [ ]:
inventory_df.to_csv(
    REPORTS_DIR / "cleaning_initial_inventory.csv",
    index=False
)

pd.DataFrame(schema_differences).to_csv(
    REPORTS_DIR / "cleaning_schema_comparison.csv",
    index=False
)

severity_df.to_csv(
    REPORTS_DIR / "cleaning_severity_values_observed.csv",
    index=False
)

print("Initial validation reports saved in:", REPORTS_DIR)

Initial validation reports saved in: /content/traffic-accident-risk-analytics-github/reports


## Step 7 - Validate crash identifiers

In [ ]:
id_results = []

for year, df in annual_data.items():
    id_col = "Crash_ID"

    if id_col not in df.columns:
        id_results.append({
            "year": year,
            "rows": len(df),
            "missing_ids": None,
            "unique_ids": None,
            "repeated_id_rows": None,
            "status": "Crash_ID column missing"
        })
        continue

    ids = df[id_col]

    id_results.append({
        "year": year,
        "rows": len(df),
        "missing_ids": int(ids.isna().sum()),
        "unique_ids": int(ids.nunique()),
        "repeated_id_rows": int(ids.duplicated(keep=False).sum()),
        "status": "Inspected"
    })

id_validation_df = pd.DataFrame(id_results)
display(id_validation_df)

id_validation_df.to_csv(
    REPORTS_DIR / "cleaning_crash_id_validation.csv",
    index=False
)

,year,rows,missing_ids,unique_ids,repeated_id_rows,status
0,2021,97947,0,97947,0,Inspected
1,2022,94700,0,94700,0,Inspected
2,2023,92082,0,92082,0,Inspected
3,2024,83092,0,83092,0,Inspected
4,2025,82670,0,82670,0,Inspected


## Step 8 — Validate crash dates

In [ ]:
date_results = []

for year, df in annual_data.items():
    raw_dates = df["Crash_Date"]

    parsed_dates = pd.to_datetime(
        raw_dates,
        errors="coerce"
    )

    # Identify non-missing source values that could not be parsed.
    invalid_dates = raw_dates.notna() & parsed_dates.isna()

    valid_dates = parsed_dates.dropna()

    date_results.append({
        "year": year,
        "rows": len(df),
        "missing_dates": int(raw_dates.isna().sum()),
        "unparseable_dates": int(invalid_dates.sum()),
        "parsed_dates": int(parsed_dates.notna().sum()),
        "minimum_date": (
            valid_dates.min().date() if not valid_dates.empty else None
        ),
        "maximum_date": (
            valid_dates.max().date() if not valid_dates.empty else None
        ),
        "dates_outside_expected_year": int(
            (parsed_dates.notna() & (parsed_dates.dt.year != year)).sum()
        )
    })

date_validation_df = pd.DataFrame(date_results)
display(date_validation_df)

date_validation_df.to_csv(
    REPORTS_DIR / "cleaning_date_validation.csv",
    index=False
)

,year,rows,missing_dates,unparseable_dates,parsed_dates,minimum_date,maximum_date,dates_outside_expected_year
0,2021,97947,0,0,97947,2021-01-01,2021-12-31,0
1,2022,94700,0,0,94700,2022-01-01,2022-12-31,0
2,2023,92082,0,0,92082,2023-01-01,2023-12-31,0
3,2024,83092,0,0,83092,2024-01-01,2024-12-31,0
4,2025,82670,0,0,82670,2025-01-01,2025-12-31,0


## Step 9 — Investigate missing values

In [ ]:
missing_results = []

for year, df in annual_data.items():
    for column in df.columns:
        missing_count = int(df[column].isna().sum())

        missing_results.append({
            "year": year,
            "column": column,
            "missing_count": missing_count,
            "missing_percent": round(
                missing_count / len(df) * 100, 2
            ) if len(df) else 0
        })

missing_df = pd.DataFrame(missing_results)

missing_df = missing_df.sort_values(
    ["year", "missing_percent"],
    ascending=[True, False]
)

display(missing_df.head(30))

missing_df.to_csv(
    REPORTS_DIR / "cleaning_missing_values.csv",
    index=False
)

print("Missing-value report saved.")

,year,column,missing_count,missing_percent
61,2021,Investigat_Area_ID,97947,100.0
62,2021,Investigat_District_ID,97947,100.0
94,2021,Street_Nbr_2,97947,100.0
106,2021,Hwy_Dsgn_Hrt_ID,97947,100.0
112,2021,Row_Width_Usual,97947,100.0
134,2021,Curve_Type_ID,97947,100.0
135,2021,Curve_Lngth,97947,100.0
136,2021,Cd_Degr,97947,100.0
137,2021,Delta_Left_Right_ID,97947,100.0
138,2021,Dd_Degr,97947,100.0


Missing-value report saved.


## Step 10 — Check the official severity lookup

In [ ]:
lookup_results = []

for year in years:
    lookup_file = RAW_DIR / str(year) / "lookup.csv"

    if not lookup_file.exists():
        print(f"{year}: lookup.csv not found")
        continue

    lookup_df = pd.read_csv(lookup_file, low_memory=False)

    print(f"\n{'=' * 60}")
    print(f"YEAR: {year}")
    print("Lookup shape:", lookup_df.shape)
    print("Columns:", lookup_df.columns.tolist())

    # Show rows mentioning severity in any text column.
    text_cols = lookup_df.select_dtypes(
        include=["object", "string"]
    ).columns

    mask = pd.Series(False, index=lookup_df.index)

    for col in text_cols:
        mask |= lookup_df[col].astype(str).str.contains(
            "severity|crash_sev",
            case=False,
            na=False
        )

    matching_rows = lookup_df.loc[mask]

    if not matching_rows.empty:
        display(matching_rows.head(30))
    else:
        print("No rows explicitly mentioning severity found.")


YEAR: 2021
Lookup shape: (15017, 3)
Columns: ['ColumnName', 'ID', 'Description']


,ColumnName,ID,Description
4545,CRASH_SEV_ID,0,UNKNOWN
4546,CRASH_SEV_ID,1,SUSPECTED SERIOUS INJURY
4547,CRASH_SEV_ID,2,SUSPECTED MINOR INJURY
4548,CRASH_SEV_ID,3,POSSIBLE INJURY
4549,CRASH_SEV_ID,4,FATAL INJURY
4550,CRASH_SEV_ID,5,NOT INJURED
8700,VEH_DAMAGE_SEVERITY_ID,0,NO DAMAGE
8701,VEH_DAMAGE_SEVERITY_ID,1,DAMAGED 1 MINIMUM
8702,VEH_DAMAGE_SEVERITY_ID,2,DAMAGED 2
8703,VEH_DAMAGE_SEVERITY_ID,3,DAMAGED 3



YEAR: 2022
Lookup shape: (15017, 3)
Columns: ['ColumnName', 'ID', 'Description']


,ColumnName,ID,Description
4545,CRASH_SEV_ID,0,UNKNOWN
4546,CRASH_SEV_ID,1,SUSPECTED SERIOUS INJURY
4547,CRASH_SEV_ID,2,SUSPECTED MINOR INJURY
4548,CRASH_SEV_ID,3,POSSIBLE INJURY
4549,CRASH_SEV_ID,4,FATAL INJURY
4550,CRASH_SEV_ID,5,NOT INJURED
8700,VEH_DAMAGE_SEVERITY_ID,0,NO DAMAGE
8701,VEH_DAMAGE_SEVERITY_ID,1,DAMAGED 1 MINIMUM
8702,VEH_DAMAGE_SEVERITY_ID,2,DAMAGED 2
8703,VEH_DAMAGE_SEVERITY_ID,3,DAMAGED 3



YEAR: 2023
Lookup shape: (15017, 3)
Columns: ['ColumnName', 'ID', 'Description']


,ColumnName,ID,Description
4545,CRASH_SEV_ID,0,UNKNOWN
4546,CRASH_SEV_ID,1,SUSPECTED SERIOUS INJURY
4547,CRASH_SEV_ID,2,SUSPECTED MINOR INJURY
4548,CRASH_SEV_ID,3,POSSIBLE INJURY
4549,CRASH_SEV_ID,4,FATAL INJURY
4550,CRASH_SEV_ID,5,NOT INJURED
8700,VEH_DAMAGE_SEVERITY_ID,0,NO DAMAGE
8701,VEH_DAMAGE_SEVERITY_ID,1,DAMAGED 1 MINIMUM
8702,VEH_DAMAGE_SEVERITY_ID,2,DAMAGED 2
8703,VEH_DAMAGE_SEVERITY_ID,3,DAMAGED 3



YEAR: 2024
Lookup shape: (15039, 3)
Columns: ['ColumnName', 'ID', 'Description']


,ColumnName,ID,Description
4554,CRASH_SEV_ID,0,UNKNOWN
4555,CRASH_SEV_ID,1,SUSPECTED SERIOUS INJURY
4556,CRASH_SEV_ID,2,SUSPECTED MINOR INJURY
4557,CRASH_SEV_ID,3,POSSIBLE INJURY
4558,CRASH_SEV_ID,4,FATAL INJURY
4559,CRASH_SEV_ID,5,NOT INJURED
8722,VEH_DAMAGE_SEVERITY_ID,0,NO DAMAGE
8723,VEH_DAMAGE_SEVERITY_ID,1,DAMAGED 1 MINIMUM
8724,VEH_DAMAGE_SEVERITY_ID,2,DAMAGED 2
8725,VEH_DAMAGE_SEVERITY_ID,3,DAMAGED 3



YEAR: 2025
Lookup shape: (15039, 3)
Columns: ['ColumnName', 'ID', 'Description']


,ColumnName,ID,Description
4554,CRASH_SEV_ID,0,UNKNOWN
4555,CRASH_SEV_ID,1,SUSPECTED SERIOUS INJURY
4556,CRASH_SEV_ID,2,SUSPECTED MINOR INJURY
4557,CRASH_SEV_ID,3,POSSIBLE INJURY
4558,CRASH_SEV_ID,4,FATAL INJURY
4559,CRASH_SEV_ID,5,NOT INJURED
8722,VEH_DAMAGE_SEVERITY_ID,0,NO DAMAGE
8723,VEH_DAMAGE_SEVERITY_ID,1,DAMAGED 1 MINIMUM
8724,VEH_DAMAGE_SEVERITY_ID,2,DAMAGED 2
8725,VEH_DAMAGE_SEVERITY_ID,3,DAMAGED 3


## Step 11 — Save the validation summary

In [ ]:
print("Validation reports saved in:")
print(REPORTS_DIR)

for filename in [
    "cleaning_initial_inventory.csv",
    "cleaning_schema_comparison.csv",
    "cleaning_severity_values_observed.csv",
    "cleaning_crash_id_validation.csv",
    "cleaning_date_validation.csv",
    "cleaning_missing_values.csv"
]:
    path = REPORTS_DIR / filename
    print(f"{filename}: {'FOUND' if path.exists() else 'NOT CREATED'}")

Validation reports saved in:
/content/traffic-accident-risk-analytics-github/reports
cleaning_initial_inventory.csv: FOUND
cleaning_schema_comparison.csv: FOUND
cleaning_severity_values_observed.csv: FOUND
cleaning_crash_id_validation.csv: FOUND
cleaning_date_validation.csv: FOUND
cleaning_missing_values.csv: FOUND


## Step 12 - Validate severity codes programmatically

In [ ]:
severity_validation = []

def normalize_code(value):
    """Normalize numeric codes for comparison without changing source data."""
    if pd.isna(value):
        return None

    value = str(value).strip()

    try:
        number = float(value)
        if number.is_integer():
            return str(int(number))
    except (ValueError, TypeError):
        pass

    return value


for year, df in annual_data.items():
    lookup_file = RAW_DIR / str(year) / "lookup.csv"
    lookup_df = pd.read_csv(lookup_file, low_memory=False)

    severity_lookup = lookup_df[
        lookup_df["ColumnName"].astype(str).str.upper().eq("CRASH_SEV_ID")
    ].copy()

    valid_codes = set(
        severity_lookup["ID"].map(normalize_code).dropna()
    )

    descriptions = {}

    for _, row in severity_lookup.iterrows():
        code = normalize_code(row["ID"])
        if code is not None:
            descriptions[code] = row["Description"]

    for value, count in df["Crash_Sev_ID"].value_counts(
        dropna=False
    ).items():
        code = normalize_code(value)

        if code is None:
            status = "Missing severity code"
            description = None
        elif code in valid_codes:
            status = "Present in CRIS lookup"
            description = descriptions.get(code)
        else:
            status = "Not found in CRIS lookup"
            description = None

        severity_validation.append({
            "year": year,
            "severity_code": code,
            "count": int(count),
            "lookup_status": status,
            "official_description": description
        })

severity_validation_df = pd.DataFrame(severity_validation)

display(
    severity_validation_df.sort_values(
        ["year", "severity_code"],
        na_position="first"
    )
)

severity_validation_df.to_csv(
    REPORTS_DIR / "cleaning_severity_validation.csv",
    index=False
)

print("Severity validation report saved.")

,year,severity_code,count,lookup_status,official_description
3,2021,0,7562,Present in CRIS lookup,UNKNOWN
4,2021,1,2631,Present in CRIS lookup,SUSPECTED SERIOUS INJURY
2,2021,2,11396,Present in CRIS lookup,SUSPECTED MINOR INJURY
1,2021,3,16235,Present in CRIS lookup,POSSIBLE INJURY
5,2021,4,550,Present in CRIS lookup,FATAL INJURY
0,2021,5,59573,Present in CRIS lookup,NOT INJURED
9,2022,0,7163,Present in CRIS lookup,UNKNOWN
10,2022,1,2520,Present in CRIS lookup,SUSPECTED SERIOUS INJURY
8,2022,2,12255,Present in CRIS lookup,SUSPECTED MINOR INJURY
7,2022,3,14780,Present in CRIS lookup,POSSIBLE INJURY


Severity validation report saved.


## Step 13 — Combine and align the five annual datasets

In [ ]:
cleaned_parts = []

for year, df in annual_data.items():
    part = df.copy()

    # Preserve the year of origin for validation and temporal analysis.
    part["Source_Year"] = year

    cleaned_parts.append(part)

cleaned_crashes = pd.concat(
    cleaned_parts,
    ignore_index=True,
    sort=False
)

print("Combined dataset shape:", cleaned_crashes.shape)
print("Total records:", f"{len(cleaned_crashes):,}")
print("Unique crash IDs:", cleaned_crashes["Crash_ID"].nunique())
print("Missing crash IDs:", cleaned_crashes["Crash_ID"].isna().sum())

print("\nRecords by source year:")
display(cleaned_crashes["Source_Year"].value_counts().sort_index())

Combined dataset shape: (450491, 184)
Total records: 450,491
Unique crash IDs: 450491
Missing crash IDs: 0

Records by source year:


,count
Source_Year,
2021,97947
2022,94700
2023,92082
2024,83092
2025,82670


## Step 14 — Standardize crash dates

In [ ]:
cleaned_crashes["Crash_Date"] = pd.to_datetime(
    cleaned_crashes["Crash_Date"],
    errors="coerce"
)

invalid_processed_dates = int(
    cleaned_crashes["Crash_Date"].isna().sum()
)

outside_year = int(
    (
        cleaned_crashes["Crash_Date"].dt.year
        != cleaned_crashes["Source_Year"]
    ).sum()
)

print("Missing or unparseable dates:", invalid_processed_dates)
print("Dates outside their source year:", outside_year)

if invalid_processed_dates > 0 or outside_year > 0:
    print(
        "WARNING: Review the date validation results before "
        "considering the dataset fully validated."
    )
else:
    print("All processed dates passed the initial checks.")

Missing or unparseable dates: 0
Dates outside their source year: 0
All processed dates passed the initial checks.


## Step 15 — Save the processed dataset

In [ ]:
from pathlib import Path
import zipfile

PROJECT_DIR = Path("/content/traffic-accident-risk-analytics-github")
ZIP_PATH = Path("/content/phase3_cleaning_outputs.zip")

files_to_include = [
    PROJECT_DIR / "data/processed/crash_cleaned.csv",
    PROJECT_DIR / "reports/cleaning_initial_inventory.csv",
    PROJECT_DIR / "reports/cleaning_schema_comparison.csv",
    PROJECT_DIR / "reports/cleaning_crash_id_validation.csv",
    PROJECT_DIR / "reports/cleaning_date_validation.csv",
    PROJECT_DIR / "reports/cleaning_missing_values.csv",
    PROJECT_DIR / "reports/cleaning_severity_validation.csv",
    PROJECT_DIR / "reports/cleaning_summary.csv",
    PROJECT_DIR / "reports/cleaning_decisions.md",
]

with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as z:
    for file in files_to_include:
        if file.exists():
            z.write(file, file.relative_to(PROJECT_DIR))
            print("Added:", file.relative_to(PROJECT_DIR))
        else:
            print("MISSING:", file.relative_to(PROJECT_DIR))

print("ZIP created:", ZIP_PATH)

Added: data/processed/crash_cleaned.csv
Added: reports/cleaning_initial_inventory.csv
Added: reports/cleaning_schema_comparison.csv
Added: reports/cleaning_crash_id_validation.csv
Added: reports/cleaning_date_validation.csv
Added: reports/cleaning_missing_values.csv
Added: reports/cleaning_severity_validation.csv
Added: reports/cleaning_summary.csv
Added: reports/cleaning_decisions.md
ZIP created: /content/phase3_cleaning_outputs.zip


In [35]:
from google.colab import files
files.download('/content/traffic-accident-risk-analytics-github/data/processed/crash_cleaned.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Step 16 — Generate the cleaning summary report

In [ ]:
cleaning_summary = []

for year in years:
    part = cleaned_crashes[
        cleaned_crashes["Source_Year"] == year
    ]

    year_severity = severity_validation_df[
        severity_validation_df["year"] == year
    ]

    cleaning_summary.append({
        "year": year,
        "original_rows": len(annual_data[year]),
        "processed_rows": len(part),
        "rows_removed": len(annual_data[year]) - len(part),
        "unique_crash_ids": int(part["Crash_ID"].nunique()),
        "missing_crash_ids": int(part["Crash_ID"].isna().sum()),
        "missing_dates": int(part["Crash_Date"].isna().sum()),
        "severity_codes_not_in_lookup": int(
            year_severity.loc[
                year_severity["lookup_status"]
                == "Not found in CRIS lookup",
                "count"
            ].sum()
        )
    })

cleaning_summary_df = pd.DataFrame(cleaning_summary)

display(cleaning_summary_df)

cleaning_summary_df.to_csv(
    REPORTS_DIR / "cleaning_summary.csv",
    index=False
)

print("Cleaning summary saved.")

,year,original_rows,processed_rows,rows_removed,unique_crash_ids,missing_crash_ids,missing_dates,severity_codes_not_in_lookup
0,2021,97947,97947,0,97947,0,0,0
1,2022,94700,94700,0,94700,0,0,0
2,2023,92082,92082,0,92082,0,0,0
3,2024,83092,83092,0,83092,0,0,0
4,2025,82670,82670,0,82670,0,0,1


Cleaning summary saved.


## Step 17 — Verify all Phase 3 deliverables

In [ ]:
deliverables = [
    PROCESSED_DIR / "crash_cleaned.csv",
    REPORTS_DIR / "cleaning_initial_inventory.csv",
    REPORTS_DIR / "cleaning_schema_comparison.csv",
    REPORTS_DIR / "cleaning_crash_id_validation.csv",
    REPORTS_DIR / "cleaning_date_validation.csv",
    REPORTS_DIR / "cleaning_missing_values.csv",
    REPORTS_DIR / "cleaning_severity_validation.csv",
    REPORTS_DIR / "cleaning_summary.csv",
    REPORTS_DIR / "cleaning_decisions.md"
]

for path in deliverables:
    status = "FOUND" if path.exists() else "MISSING"
    print(f"{status}: {path.relative_to(PROJECT_DIR)}")

print("\nFinal combined shape:", cleaned_crashes.shape)
print("Total rows:", f"{len(cleaned_crashes):,}")
print("Total columns:", len(cleaned_crashes.columns))

FOUND: data/processed/crash_cleaned.csv
FOUND: reports/cleaning_initial_inventory.csv
FOUND: reports/cleaning_schema_comparison.csv
FOUND: reports/cleaning_crash_id_validation.csv
FOUND: reports/cleaning_date_validation.csv
FOUND: reports/cleaning_missing_values.csv
FOUND: reports/cleaning_severity_validation.csv
FOUND: reports/cleaning_summary.csv
MISSING: reports/cleaning_decisions.md

Final combined shape: (450491, 184)
Total rows: 450,491
Total columns: 184


## Step 18 — Documentation file

In [ ]:
from pathlib import Path

decisions_file = REPORTS_DIR / "cleaning_decisions.md"

decisions_text = """
# Data Cleaning and Preparation Decisions

## Dataset
Texas CRIS crash records for Dallas and Tarrant counties, 2021–2025.

## Data integrity checks
- Combined annual datasets while preserving the union of available columns.
- Added Source_Year to identify the source year of each record.
- Validated crash IDs and crash dates.
- Compared observed severity codes against the CRIS lookup definitions.
- Preserved raw data without modifying the original annual CSV files.

## Cleaning decisions
- Retained all 450,491 crash records.
- No rows were removed during this stage.
- Standardized Crash_Date to a datetime representation.
- Preserved columns introduced in later annual schemas.
- Retained the 2025 record with severity code 95 for investigation.
  This code was not found in the standard CRIS severity lookup, so it
  has not been silently deleted or recoded.
- Missing values are documented for further investigation rather than
  automatically filled or treated as zero.

## Important modeling consideration
Crash severity is the prediction target. Outcome-derived fields, such as
fatality indicators and injury counts, must not be used as predictors
when they would reveal the outcome.
"""

decisions_file.write_text(decisions_text.strip() + "\n", encoding="utf-8")

print("Saved:", decisions_file)
print("File exists:", decisions_file.exists())

Saved: /content/traffic-accident-risk-analytics-github/reports/cleaning_decisions.md
File exists: True


## Step 19 — Verify the final dataset and summary

In [ ]:
print("Dataset shape:", cleaned_crashes.shape)
print("Total rows:", len(cleaned_crashes))
print("Unique crash IDs:", cleaned_crashes["Crash_ID"].nunique())
print("Missing crash IDs:", cleaned_crashes["Crash_ID"].isna().sum())
print("Missing crash dates:", cleaned_crashes["Crash_Date"].isna().sum())

display(cleaning_summary_df)

print("Cleaning decisions file exists:", decisions_file.exists())

Dataset shape: (450491, 184)
Total rows: 450491
Unique crash IDs: 450491
Missing crash IDs: 0
Missing crash dates: 0


,year,original_rows,processed_rows,rows_removed,unique_crash_ids,missing_crash_ids,missing_dates,severity_codes_not_in_lookup
0,2021,97947,97947,0,97947,0,0,0
1,2022,94700,94700,0,94700,0,0,0
2,2023,92082,92082,0,92082,0,0,0
3,2024,83092,83092,0,83092,0,0,0
4,2025,82670,82670,0,82670,0,0,1


Cleaning decisions file exists: True
